In [1]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, Subset

from CLAPWrapper import CLAPWrapper
from datasets.esc50 import ESC50
from datasets.vocalsound import VocalSound
from datasets.tinysol import TinySOL
from datasets.irmas import IRMAS

# ============================================================================
# CONFIG — cambia qui per switchare dataset e iperparametri
# ============================================================================

DATASET         = TinySOL
DATASET_ROOT    = "../data"
VERSION         = '2023'

# Config ResiDual v3:
#   target_layers scelti empiricamente dai dati (blocchi 0-1 degeneri → esclusi)
#   variance_threshold: 0.95 = pca_95, 0.90 = pca_90 (più aggressivo)
RESIDUAL_CONFIG = {
    'target_layers':      [2, 3],
    'variance_threshold': 0.95,
}

PCA_SAMPLES  = 200   # campioni per fit_pca_on_data
TEST_SIZE    = 300   # campioni per valutazione accuracy
MAX_EPOCHS   = 10    # max epoche ottimizzazione lambda
PATIENCE     = 3     # early stopping patience
LR           = 1e-2  # learning rate Schedule-Free Adam

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

torch.manual_seed(0)
torch.cuda.manual_seed(0)
np.random.seed(0)

Device: cpu


In [2]:
# ============================================================================
# DATASET
# ============================================================================
print(f"\nCaricamento {DATASET.__name__}...")
dataset    = DATASET(root=DATASET_ROOT, download=False)
n_classes  = len(dataset.classes)
print(f"  Samples  : {len(dataset)}")
print(f"  Classi   : {n_classes} → {dataset.classes}")

prompt      = 'this is the sound of '
text_labels = [prompt + x for x in dataset.classes]


Caricamento TinySOL...
Loading audio files


 43%|████▎     | 1249/2913 [00:00<00:00, 6242.09it/s]

100%|██████████| 2913/2913 [00:00<00:00, 6712.29it/s]

✓ Cache di validazione trovata (2026-04-01T23:54:22.189250): 2913 validi, 0 corrotti. Skip validazione.
  Samples  : 2913
  Classi   : 14 → ['Accordion', 'Bass Tuba', 'Bassoon', 'Clarinet Bb', 'Contrabass', 'Flute', 'Horn', 'Oboe', 'Sax Alto', 'Trombone', 'Trumpet C', 'Viola', 'Violin', 'Violoncello']


In [3]:
# ============================================================================
# MODELLI
# ============================================================================
print("\nCaricamento CLAP standard (baseline)...", end='')
clap_standard = CLAPWrapper(
    version  = VERSION,
    use_cuda = torch.cuda.is_available(),
    type     = 'classic'
)
print('OK')

print("Caricamento ResiDual CLAP v3...", end='')
clap_residual = CLAPWrapper(
    version         = VERSION,
    use_cuda        = torch.cuda.is_available(),
    type            = 'residual',
    residual_config = RESIDUAL_CONFIG
)
print('OK')


Caricamento CLAP standard (baseline)...OK
Caricamento ResiDual CLAP v3...OK


In [4]:
# ============================================================================
# DATASET AUSILIARI
# ============================================================================

class SimpleAudioDataset(torch.utils.data.Dataset):
    """Dataset minimale che restituisce tensori audio grezzi."""

    def __init__(self, wrapper, dataset, max_samples: int = 200):
        self.wrapper     = wrapper
        self.audio_paths = []

        # Campionamento stratificato per classe
        samples_per_class = max_samples // len(dataset.classes)
        class_buckets     = {cls: [] for cls in dataset.classes}

        for i in range(len(dataset)):
            audio_path, class_name, _ = dataset[i]
            if len(class_buckets[class_name]) < samples_per_class:
                class_buckets[class_name].append(audio_path)

        for paths in class_buckets.values():
            self.audio_paths.extend(paths)

    def __len__(self):
        return len(self.audio_paths)

    def __getitem__(self, idx):
        audio = self.wrapper.load_audio_into_tensor(
            self.audio_paths[idx],
            self.wrapper.args.duration,
            resample=True
        )
        if audio.dim() > 1:
            audio = audio.squeeze()
        return audio


class LabeledAudioDataset(torch.utils.data.Dataset):
    """
    Dataset che restituisce (audio_tensor, label_int).
    Usato per optimize_spectral_weights.
    """

    def __init__(self, wrapper, dataset, max_samples: int = None):
        self.wrapper     = wrapper
        self.samples     = []   # lista di (audio_path, label_idx)

        classes = dataset.classes

        for i in range(len(dataset)):
            audio_path, class_name, _ = dataset[i]
            label_idx = classes.index(class_name)
            self.samples.append((audio_path, label_idx))

        if max_samples is not None:
            # Campionamento stratificato
            from collections import defaultdict
            buckets = defaultdict(list)
            for path, label in self.samples:
                buckets[label].append((path, label))
            per_class = max_samples // len(classes)
            self.samples = []
            for items in buckets.values():
                self.samples.extend(items[:per_class])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        audio_path, label = self.samples[idx]
        audio = self.wrapper.load_audio_into_tensor(
            audio_path,
            self.wrapper.args.duration,
            resample=True
        )
        if audio.dim() > 1:
            audio = audio.squeeze()
        return audio, torch.tensor(label, dtype=torch.long)

In [5]:
# ============================================================================
# BASELINE — CLAP standard
# ============================================================================
print(f"\nBaseline CLAP su {TEST_SIZE} campioni...")

text_embs_standard = clap_standard.get_text_embeddings(text_labels)

y_preds_baseline, y_labels_baseline = [], []

for i in tqdm(range(TEST_SIZE), desc="Baseline"):
    audio_path, class_name, one_hot = dataset[-(i + 1)]
    audio_emb  = clap_standard.get_audio_embeddings([audio_path], resample=True)
    similarity = clap_standard.compute_similarity(audio_emb, text_embs_standard)
    y_pred     = F.softmax(similarity.detach().cpu(), dim=1).numpy()
    y_preds_baseline.append(y_pred)
    y_labels_baseline.append(one_hot.detach().cpu().numpy())

y_labels_arr   = np.concatenate(y_labels_baseline, axis=0)
y_preds_bl_arr = np.concatenate(y_preds_baseline,  axis=0)
baseline_acc   = accuracy_score(
    np.argmax(y_labels_arr,   axis=1),
    np.argmax(y_preds_bl_arr, axis=1)
)
print(f"\n✅ Baseline Accuracy: {baseline_acc:.3f} ({baseline_acc*100:.1f}%)")


Baseline CLAP su 300 campioni...


Baseline:   0%|          | 0/300 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [5]:
# ============================================================================
# FASE 1 — fit_pca_on_data
#   Raccoglie hidden states degli heads target e calcola Φ, μ per ogni head.
#   Inizializza λ: 1 per le prime k PC, 0 per le restanti.
# ============================================================================
print(f"\nFase 1: fit PCA su {PCA_SAMPLES} campioni...")

pca_dataset = SimpleAudioDataset(clap_residual, dataset, max_samples=PCA_SAMPLES)
pca_loader  = DataLoader(pca_dataset, batch_size=1, shuffle=False, num_workers=0)

fit_info = clap_residual.clap.fit_spectral_components(
    pca_loader,
    max_samples=PCA_SAMPLES
)

# Log compatto per ogni head
print("\nRisultati fit PCA per head:")
for layer_name, layer_data in fit_info.items():
    for block_name, block_data in layer_data.items():
        for head_name, info in block_data.items():
            print(f"  {layer_name} | {block_name} | {head_name}: "
                  f"k={info['k']:3d} | var@k={info['variance_at_k']:.3f} | "
                  f"EVR_pc1={info['evr_pc1']:.3f}")


Fase 1: fit PCA su 200 campioni...


[ResiDual] Raccolta per PCA: 100%|██████████| 196/196 [00:42<00:00,  4.65it/s]


[ResiDual] Raccolti 196 campioni
  layer_2 | block 0 | head_0: k=12 | var@k=0.962
  layer_2 | block 0 | head_1: k=15 | var@k=0.954
  layer_2 | block 0 | head_2: k=15 | var@k=0.950
  layer_2 | block 0 | head_3: k=13 | var@k=0.951
  layer_2 | block 0 | head_4: k=12 | var@k=0.954
  layer_2 | block 0 | head_5: k=10 | var@k=0.966
  layer_2 | block 0 | head_6: k=11 | var@k=0.957
  layer_2 | block 0 | head_7: k=10 | var@k=0.952
  layer_2 | block 0 | head_8: k=17 | var@k=0.958
  layer_2 | block 0 | head_9: k=16 | var@k=0.959
  layer_2 | block 0 | head_10: k=13 | var@k=0.958
  layer_2 | block 0 | head_11: k=12 | var@k=0.955
  layer_2 | block 0 | head_12: k=12 | var@k=0.951
  layer_2 | block 0 | head_13: k=12 | var@k=0.953
  layer_2 | block 0 | head_14: k=15 | var@k=0.951
  layer_2 | block 0 | head_15: k=14 | var@k=0.958
  layer_2 | block 1 | head_0: k=14 | var@k=0.951
  layer_2 | block 1 | head_1: k=17 | var@k=0.960
  layer_2 | block 1 | head_2: k=18 | var@k=0.954
  layer_2 | block 1 | head_3: 

In [6]:
# ============================================================================
# FASE 2 — optimize_spectral_weights
#   Ottimizza λ per massimizzare zero-shot accuracy sul validation set.
#   Solo i λ vengono aggiornati — Φ, μ e il resto del modello sono congelati.
# ============================================================================
print(f"\nFase 2: ottimizzazione λ (max {MAX_EPOCHS} epoche, patience={PATIENCE})...")

# Validation set: usa gli stessi campioni del test (split separato in produzione)
val_dataset = LabeledAudioDataset(clap_residual, dataset, max_samples=PCA_SAMPLES)
val_loader  = DataLoader(val_dataset, batch_size=1, shuffle=True, num_workers=0)

# Text embeddings per le classi — calcolati una volta sola con il modello residual
# (lo stesso encoder testuale di CLAP standard)
text_embs_residual = clap_residual.get_text_embeddings(text_labels)  # [n_classes, d_proj]

history = clap_residual.clap.optimize_spectral_weights(
    val_dataloader        = val_loader,
    class_text_embeddings = text_embs_residual,
    max_epochs            = MAX_EPOCHS,
    patience              = PATIENCE,
    lr                    = LR,
)

print(f"\nOttimizzazione completata:")
print(f"  Best accuracy (val): {history['best_accuracy']:.3f} @ epoch {history['best_epoch']}")
print(f"  Curva accuracy: {[f'{a:.3f}' for a in history['accuracy']]}")


Fase 2: ottimizzazione λ (max 10 epoche, patience=3)...


KeyboardInterrupt: 

In [7]:
val_dataset = LabeledAudioDataset(clap_residual, dataset, max_samples=PCA_SAMPLES)
val_loader  = DataLoader(val_dataset, batch_size=8, shuffle=True, num_workers=0)

In [8]:
for batch in val_loader:
    h = batch
    break

In [10]:
h[0].shape

torch.Size([8, 308700])

In [7]:
# ============================================================================
# VALUTAZIONE — ResiDual v3
# ============================================================================
print(f"\nResiDual CLAP v3 su {TEST_SIZE} campioni...")

y_preds_residual, y_labels_residual = [], []

for i in tqdm(range(TEST_SIZE), desc="ResiDual v3"):
    audio_path, class_name, one_hot = dataset[-(i + 1)]
    audio_emb  = clap_residual.get_audio_embeddings([audio_path], resample=True)
    similarity = clap_residual.compute_similarity(audio_emb, text_embs_residual)
    y_pred     = F.softmax(similarity.detach().cpu(), dim=1).numpy()
    y_preds_residual.append(y_pred)
    y_labels_residual.append(one_hot.detach().cpu().numpy())

y_labels_arr    = np.concatenate(y_labels_residual, axis=0)
y_preds_res_arr = np.concatenate(y_preds_residual,  axis=0)
residual_acc    = accuracy_score(
    np.argmax(y_labels_arr,    axis=1),
    np.argmax(y_preds_res_arr, axis=1)
)
print(f"\n✅ ResiDual v3 Accuracy: {residual_acc:.3f} ({residual_acc*100:.1f}%)")


ResiDual CLAP v3 su 300 campioni...


ResiDual v3:   0%|          | 0/300 [00:00<?, ?it/s]


✅ ResiDual v3 Accuracy: 0.590 (59.0%)


In [ ]:
# ============================================================================
# RIEPILOGO
# ============================================================================
delta = residual_acc - baseline_acc
print("\n" + "="*50)
print(f"  Dataset          : {DATASET.__name__}")
print(f"  Target layers    : {RESIDUAL_CONFIG['target_layers']}")
print(f"  Var. threshold   : {RESIDUAL_CONFIG['variance_threshold']} (pca_{int(RESIDUAL_CONFIG['variance_threshold']*100)})")
print(f"  Test samples     : {TEST_SIZE}")
print(f"  Baseline CLAP    : {baseline_acc:.3f} ({baseline_acc*100:.1f}%)")
print(f"  ResiDual v3      : {residual_acc:.3f} ({residual_acc*100:.1f}%)")
print(f"  Delta            : {delta:+.3f} ({delta*100:+.1f}%)")
print("="*50)


  Dataset          : ESC50
  Target layers    : [1, 2, 3]
  Var. threshold   : 0.9 (pca_90)
  Test samples     : 300
  Baseline CLAP    : 0.933 (93.3%)
  ResiDual v3      : 0.943 (94.3%)
  Delta            : +0.010 (+1.0%)
